# BiLSTM with Attention (From Scratch) 

This notebook implements a bidirectional LSTM with an attention mechanism, built entirely
from scratch (no pretrained weights or embeddings), to predict the correct answer among 5
options for each MCQ prompt. It outputs the top 3 probable options for each question.

# Importing Libraries

In [1]:
import os
import re
import random
import numpy as np
import pandas as pd
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb
from kaggle_secrets import UserSecretsClient
from dataclasses import dataclass, asdict

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style="whitegrid", palette="muted")

Using device: cuda


# Loading Dataset

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print("Datasets loaded!")

Datasets loaded!


# Define the Model

## Configuration

In [3]:
@dataclass
class Config:
    model_name: str = "bilstm-simple-attention"
    vocab_size: int = 25000
    embed_dim: int = 128
    hidden_dim: int = 256
    max_seq_len: int = 200
    
    epochs: int = 10
    batch_size: int = 32  
    learning_rate: float = 2e-3 
    weight_decay: float = 0.05

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    wandb_project: str = "23f2004791-t22026"
    wandb_run_name: str = "bilstm-simple-attention"
    
    def to_dict(self):
        return asdict(self)

cfg = Config()

In [4]:
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
    print("Successfully logged into Weights & Biases!")
except Exception as e:
    print(f"W&B Login Failed.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adrija935 (23f2004791-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged into Weights & Biases!


## Tokenizer and Vocabulary Builder

In [5]:
class SimpleVocab:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1} 
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.min_freq = min_freq
        self.vocab_size = 2

    def tokenize(self, text):
        # Lowercase and remove punctuation
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return text.split()

    def build_vocab(self, texts):
        word_counts = Counter()
        for text in texts:
            word_counts.update(self.tokenize(text))
            
        for word, count in word_counts.items():
            if count >= self.min_freq:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1

    def encode(self, text, max_len=128):
        tokens = self.tokenize(text)
        ids = [self.word2idx.get(word, 1) for word in tokens] 
        if len(ids) > max_len:
            ids = ids[:max_len] # Clipping
        else:
            ids = ids + [0] * (max_len - len(ids)) # Padding
        return ids


# Collect all text to build vocabulary
corpus = train_df['prompt'].tolist()
for col in ['A', 'B', 'C', 'D', 'E']:
    corpus.extend(train_df[col].tolist())
    
vocab = SimpleVocab(min_freq=2)
vocab.build_vocab(corpus)
print(f"Vocabulary size built: {vocab.vocab_size} unique tokens.")

Vocabulary size built: 3081 unique tokens.


## PyTorch Dataset and DataLoaders

In [6]:
class MCQDataset(Dataset): 
    def __init__(self, df, vocab, max_len=128, is_test=False):
        self.df = df
        self.vocab = vocab 
        self.max_len = max_len
        self.is_test = is_test
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        options = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
        
        input_ids = []
        for opt in options:
            # Combine prompt and option text 
            text = prompt + " " + opt
            ids = self.vocab.encode(text, max_len=self.max_len)
            input_ids.append(ids)
            
        input_tensor = torch.tensor(input_ids, dtype=torch.long) 
        if not self.is_test:
            label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            return input_tensor, label
        return input_tensor

train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=SEED)

print(f"Train size: {len(train_split)}")
print(f"Val size: {len(val_split)}")

# Instantiate Datasets and DataLoaders
train_dataset = MCQDataset(train_split, vocab, max_len=cfg.max_seq_len)
val_dataset = MCQDataset(val_split, vocab, max_len=cfg.max_seq_len)
test_dataset = MCQDataset(test_df, vocab, max_len=cfg.max_seq_len, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)

Train size: 1600
Val size: 400


## Model Architecture

In [7]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_out):
        scores = self.attention(lstm_out)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights * lstm_out, dim=1) 
        return context

class MCQBiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super(MCQBiLSTMAttention, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        
        self.attention = Attention(hidden_dim)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        batch_size, num_options, max_len = x.shape
        x = x.view(batch_size * num_options, max_len) 
        embedded = self.embedding(x) 
        lstm_out, _ = self.lstm(embedded)
        attended_vector = self.attention(lstm_out) 
        scores = self.fc(attended_vector) 
        logits = scores.view(batch_size, num_options) 
        return logits

## Scoring Metric (MAP@3)

In [8]:
# MAP@3 Metric
def compute_map_at_3(predictions, targets):
    scores = []
    for top_preds, target in zip(predictions, targets):
        score = 0.0
        for rank, pred in enumerate(top_preds):
            if pred == target:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

# Training and Validation

In [9]:
wandb.init(project=cfg.wandb_project, name=cfg.wandb_run_name, config=cfg.to_dict(), reinit=True)

model = MCQBiLSTMAttention(vocab_size=vocab.vocab_size, hidden_dim=cfg.hidden_dim, embed_dim=cfg.embed_dim).to(cfg.device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-5)

best_map3 = 0.0

for epoch in range(cfg.epochs):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}")
    for inputs, labels in loop:
        inputs, labels = inputs.to(cfg.device), labels.to(cfg.device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping prevents gradient saturation/explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        
    scheduler.step()
    train_loss = running_loss / len(train_loader.dataset)
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    all_top3_preds = []
    all_top1_preds = []
    all_targets = []
    
    with torch.no_grad():
        val_loop = tqdm(val_loader)
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(cfg.device), labels.to(cfg.device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)

            top1_preds = torch.argmax(outputs, dim=1)
            _, top3_indices = torch.topk(outputs, k=3, dim=1)

            all_top1_preds.append(top1_preds.cpu().numpy())
            all_top3_preds.append(top3_indices.cpu().numpy())
            all_targets.append(labels.cpu().numpy())
            
    val_loss = val_loss / len(val_loader.dataset)
    all_top1_preds = np.concatenate(all_top1_preds, axis=0)
    all_top3_preds = np.concatenate(all_top3_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    val_map3 = compute_map_at_3(all_top3_preds, all_targets)
    val_accuracy = accuracy_score(all_targets, all_top1_preds)
    val_f1 = f1_score(all_targets, all_top1_preds, average='macro')

    print(f"Epoch {epoch+1}/{cfg.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val MAP@3: {val_map3:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_f1:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_map3": val_map3,
        "val_accuracy": val_accuracy,
        "val_f1": val_f1
    })

    if val_map3 > best_map3:
        best_map3 = val_map3
        torch.save(model.state_dict(), "best_bilstm_attn_model.pt")
        wandb.run.summary["best_map3"] = best_map3
        wandb.run.summary["best_accuracy"] = val_accuracy
        wandb.run.summary["best_f1"] = val_f1
        print(f"New best model saved with MAP@3: {best_map3:.4f}")

wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run sihmilwr
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260803_160442-sihmilwr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm-simple-attention
wandb: ⭐️ View project at https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026
wandb: 🚀 View run at https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026/runs/sihmilwr
100%|██████████| 13/13 [00:00<00:00, 16.33it/s]


Epoch 1/10 | Train Loss: 1.2806 | Val Loss: 0.8227 | Val MAP@3: 0.7550 | Val Acc: 0.6675 | Val F1: 0.6572
New best model saved with MAP@3: 0.7550


100%|██████████| 13/13 [00:00<00:00, 17.80it/s]


Epoch 2/10 | Train Loss: 0.4490 | Val Loss: 0.3302 | Val MAP@3: 0.9125 | Val Acc: 0.8900 | Val F1: 0.8809
New best model saved with MAP@3: 0.9125


100%|██████████| 13/13 [00:00<00:00, 17.37it/s]


...

100%|██████████| 13/13 [00:00<00:00, 15.27it/s]


Epoch 9/10 | Train Loss: 0.0003 | Val Loss: 0.0006 | Val MAP@3: 1.0000 | Val Acc: 1.0000 | Val F1: 1.0000


100%|██████████| 13/13 [00:00<00:00, 14.54it/s]
wandb: updating run metadata


Epoch 10/10 | Train Loss: 0.0010 | Val Loss: 0.0005 | Val MAP@3: 1.0000 | Val Acc: 1.0000 | Val F1: 1.0000


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 9-9, summary, console lines 32-34
wandb: 
wandb: Run history:
wandb:        epoch ▁▂▃▃▄▅▆▆▇█
wandb:   train_loss █▃▂▁▁▁▁▁▁▁
wandb: val_accuracy ▁▆▇███████
wandb:       val_f1 ▁▆▇███████
wandb:     val_loss █▄▂▂▁▁▁▁▁▁
wandb:     val_map3 ▁▆▇███████
wandb: 
wandb: Run summary:
wandb: best_accuracy 1
wandb:       best_f1 1
wandb:     best_map3 1
wandb:         epoch 10
wandb:    train_loss 0.00098
wandb:  val_accuracy 1
wandb:        val_f1 1
wandb:      val_loss 0.00047
wandb:      val_map3 1
wandb: 
wandb: 🚀 View run bilstm-simple-attention at: https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026/runs/sihmilwr
wandb: ⭐️ View project at: https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wand

# Inference

In [10]:
model.load_state_dict(torch.load("best_bilstm_attn_model.pt"))
model.eval()

idx_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
submission_preds = []

with torch.no_grad():
    for inputs in test_loader:
        inputs = inputs.to(cfg.device)
        outputs = model(inputs)
        _, top3_indices = torch.topk(outputs, k=3, dim=1)
        
        for top_three in top3_indices.cpu().numpy():
            pred_str = " ".join([idx_to_label[i] for i in top_three])
            submission_preds.append(pred_str)

# Generate Submission

In [11]:
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': submission_preds
})

submission_df.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'!")
print(submission_df.head())

Submission saved to 'submission.csv'!
   id Prediction
0   1      A C D
1   2      B C A
2   3      B E C
3   4      E C A
4   5      C A D
